<a href="https://colab.research.google.com/github/yaesur/business_python/blob/%EC%B2%AD%EB%85%84%EB%A7%A4%EC%9E%85%EC%9E%84%EB%8C%80%EC%A3%BC%ED%83%9D/%EC%A7%80%EC%97%AD%EC%84%A0%ED%98%B8%EB%8F%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. 지역 등급 구조 정의
# '중구', '서대문구', '종로구' A등급으로 옮김
# '강동구'를 B등급으로 옮김
grade_map = {
    'S': ['강남구', '서초구', '송파구', '마포구'],
    'A': ['성동구', '용산구', '광진구', '영등포구', '동작구', '중구', '서대문구', '종로구'],
    'B': ['강동구', '성북구', '양천구', '강서구', '동대문구'],
    'C': ['도봉구', '노원구', '강북구', '중랑구', '금천구', '관악구', '구로구', '은평구']
}

file = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file)
sheets = xls.sheet_names

# 시트별 컬럼 설정
level_cols = [5, 5, 5, 5, 5, 5, 5, 4, 4, 7]
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
result = []

# 데이터 로드
for i, name in enumerate(sheets):
    df = pd.read_excel(file, sheet_name=name, skiprows=2, header=None)
    data = df.iloc[:, [1, level_cols[i], score_cols[i]]]
    data.columns = ['자치구', '순위', '점수']
    result.append(data)

final = pd.concat(result).dropna()

# 데이터 정제 (숫자형 변환)
final['순위'] = final['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
final['순위'] = pd.to_numeric(final['순위'], errors='coerce')
final['점수'] = final['점수'].astype(str).str.replace('점', '', regex=False).str.strip()
final['점수'] = pd.to_numeric(final['점수'], errors='coerce')
final['자치구'] = final['자치구'].str.strip()

final = final.dropna(subset=['순위', '점수'])

# 등급 매핑 함수
def get_grade(district):
    for grade, districts in grade_map.items():
        if district in districts:
            return grade
    return '미분류'

final['등급'] = final['자치구'].apply(get_grade)


# ========================================================
# 🎯 [순위별 오프셋 루프] 1순위, 2순위 각각 상관관계 분석
# ========================================================

grade_to_num = {'S': 3, 'A': 2, 'B': 1, 'C': 0}

for rank in [1, 2, 3]:
    print(f"\n==================================================")
    print(f"         🎯 [{rank}순위 모집 단위] 상관관계 분석         ")
    print(f"==================================================")

    # 해당 순위 데이터만 필터링
    rank_df = final[final['순위'] == rank].copy()

    if rank_df.empty:
        print(f"❌ {rank}순위 커트라인 데이터가 존재하지 않습니다.")
        continue

    # 1. 등급별 평균 점수 계산
    grade_avg = rank_df.groupby('등급')['점수'].mean().round(2).reset_index()

    # 2. 통계 분석을 위한 등급 숫자 인코딩
    grade_avg['등급_숫자'] = grade_avg['등급'].map(grade_to_num)
    grade_avg = grade_avg.dropna(subset=['등급_숫자'])

    # 가독성을 위한 정렬 (S -> A -> B -> C 순서)
    grade_avg['등급'] = pd.Categorical(grade_avg['등급'], categories=['S', 'A', 'B', 'C'], ordered=True)
    grade_avg = grade_avg.sort_values('등급')

    print(f"📊 [{rank}순위] 등급별 평균 커트라인 결과")
    print(grade_avg[['등급', '등급_숫자', '점수']].to_string(index=False))
    print("-" * 50)

    # 데이터 개수가 부족해 상관계수 계산이 불가능한 경우 예외 처리
    if len(grade_avg) > 1:
        # 3. 등급_숫자와 점수 간의 피어슨 상관계수 계산
        correlation = grade_avg['등급_숫자'].corr(grade_avg['점수'])
        print(f"📈 [{rank}순위] 선호도 등급과 커트라인의 상관계수: {correlation:.4f}")

        # 4. 상관계수 해석 출력
        if correlation >= 0.9:
            print(">>> [해석] 상관계수가 극도로 높습니다. 배치한 등급이 완벽하게 들어맞습니다.")
        elif correlation >= 0.7:
            print(">>> [해석] 강한 양의 상관관계입니다. 등급이 높을수록 컷이 뚜렷하게 올라갑니다.")
        elif correlation >= 0.4:
            print(">>> [해석] 약한 양의 상관관계입니다. 경향성은 있으나 일부 구의 조정이 필요할 수 있습니다.")
        else:
            print(">>> [해석] 상관관계가 낮거나 없습니다. 등급 조정을 재검토해야 합니다.")
    else:
        print("⚠ 등급별 데이터가 부족하여 상관계수를 계산할 수 없습니다.")

    print("==================================================")


         🎯 [1순위 모집 단위] 상관관계 분석         
📊 [1순위] 등급별 평균 커트라인 결과
등급  등급_숫자   점수
 S      3 6.46
 A      2 6.17
 B      1 5.69
 C      0 5.51
--------------------------------------------------
📈 [1순위] 선호도 등급과 커트라인의 상관계수: 0.9867
>>> [해석] 상관계수가 극도로 높습니다. 배치한 등급이 완벽하게 들어맞습니다.

         🎯 [2순위 모집 단위] 상관관계 분석         
📊 [2순위] 등급별 평균 커트라인 결과
등급  등급_숫자   점수
 S      3 6.45
 A      2 6.84
 B      1 6.06
 C      0 5.92
--------------------------------------------------
📈 [2순위] 선호도 등급과 커트라인의 상관계수: 0.7386
>>> [해석] 강한 양의 상관관계입니다. 등급이 높을수록 컷이 뚜렷하게 올라갑니다.

         🎯 [3순위 모집 단위] 상관관계 분석         
📊 [3순위] 등급별 평균 커트라인 결과
등급  등급_숫자   점수
 B      1 5.00
 C      0 5.53
--------------------------------------------------
📈 [3순위] 선호도 등급과 커트라인의 상관계수: -1.0000
>>> [해석] 상관관계가 낮거나 없습니다. 등급 조정을 재검토해야 합니다.


In [1]:
import pandas as pd

# 지역을 등급별로 구분
# '중구', '서대문구', '종로구' A등급으로 옮김
# '강동구'를 B등급으로 옮김
grade_map = {
  'S': ['강남구', '서초구', '송파구', '마포구'],
  'A': ['성동구', '용산구', '광진구', '영등포구', '동작구','중구', '서대문구', '종로구'],
  'B': ['강동구', '성북구', '양천구', '강서구', '동대문구'],
  'C': ['도봉구', '노원구', '강북구', '중랑구', '금천구', '관악구', '구로구', '은평구']
}

file = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file)
sheets = xls.sheet_names

level_cols = [5, 5, 5, 5, 5, 5, 5, 4, 4, 7]
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
result = []

# 매 시트를 위 컬럼대로 불러옴 3행부터 데이터 시작이기 때문에 skiprows = 2
for i, name in enumerate(sheets):
  df = pd.read_excel(file, sheet_name=name, skiprows=2, header=None)

  data = df.iloc[:, [1, level_cols[i], score_cols[i]]]
  data.columns = ['자치구', '순위', '점수']
result.append(data)

final = pd.concat(result).dropna()
final
# 순위와 점수컬럼에서 '순위'와 '점' 단위를 제거하고 수치형으로 바꿈
final['순위'] = final['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
final['순위'] = pd.to_numeric(final['순위'], errors='coerce')
final['점수'] = final['점수'].astype(str).str.replace('점', '', regex=False).str.strip()
final['점수'] = pd.to_numeric(final['점수'], errors='coerce')

# '순위'가 1인 행을 찾아서, 그 행들의 '점수' 컬럼에만 14를 더함
final.loc[final['순위'] == 1, '점수'] += 14

final['자치구'] = final['자치구'].str.strip()

# 등급과 등급 내 지역들을 1대1로 연계
def get_grade(district):
  for grade, districts in grade_map.items():
    if district in districts:
      return grade

final['등급'] = final['자치구'].apply(get_grade)

# 지역등급에 해당하는 커트라인의 점수 평균을 계산
grade_avg = final.groupby('등급')['점수'].mean().round(2).reset_index()

print(grade_avg)
data = {'등급_숫자': [3, 2, 1, 0], '점수': [17.91, 16.17, 13.82, 12.08]} # S=3, A=2, B=1, C=0
df_corr = pd.DataFrame(data)

correlation = df_corr['등급_숫자'].corr(df_corr['점수'])

print(f"상관계수: {correlation:.4f}")
print("1에 가까울수록 등급과 점수가 정확히 비례한다는 뜻입니다.")

FileNotFoundError: [Errno 2] No such file or directory: '커트라인 데이터.xlsx'